In [3]:
import pandas as pd

# Load parquet
df = pd.read_parquet("crsp_top500_daily.parquet", engine="pyarrow")

# Make sure date column is datetime
df["DlyCalDt"] = pd.to_datetime(df["DlyCalDt"])

# Keep only 1993 to and including 2003
df_1993_2003 = df[
    (df["DlyCalDt"] >= "1993-01-01") &
    (df["DlyCalDt"] <= "2003-12-31")
].copy()

# Drop duplicates based on stock id + date
df_1993_2003 = df_1993_2003.drop_duplicates(
    subset=["PERMNO", "DlyCalDt"],
    keep="first"
).copy()

# Check result
print(df_1993_2003.head())
print(df_1993_2003.shape)
print(df_1993_2003["DlyCalDt"].min(), "to", df_1993_2003["DlyCalDt"].max())

# Save filtered data
df_1993_2003.to_csv("crsp_top500_daily_1993_2003.csv", index=False)

    DlyCalDt  PERMNO      DlyCap  DlyOpen  DlyHigh  DlyLow  DlyClose    DlyVol
0 1993-01-04   10137  2703337.88   47.625   47.625  47.250    47.625   99000.0
1 1993-01-05   10137  2696242.50   47.500   47.625  47.250    47.500  124300.0
2 1993-01-06   10137  2696242.50   47.625   47.625  47.375    47.500   65900.0
3 1993-01-07   10137  2696242.50   47.500   47.625  47.500    47.500  103400.0
4 1993-01-08   10137  2674956.38   47.375   47.500  46.875    47.125   74000.0
(1386000, 8)
1993-01-04 00:00:00 to 2003-12-31 00:00:00


In [4]:
import pandas as pd

# Load csv
df = pd.read_csv("crsp_top500_daily_1993_2003.csv")

# Make sure date is datetime
df["DlyCalDt"] = pd.to_datetime(df["DlyCalDt"])

# Check duplicates on PERMNO + date
dups = df[df.duplicated(subset=["PERMNO", "DlyCalDt"], keep=False)].copy()

print("Number of duplicated rows:", len(dups))
print("Number of duplicated PERMNO-date combinations:", dups[["PERMNO", "DlyCalDt"]].drop_duplicates().shape[0])

# Show a few examples
print(dups.sort_values(["PERMNO", "DlyCalDt"]).head(20))

Number of duplicated rows: 0
Number of duplicated PERMNO-date combinations: 0
Empty DataFrame
Columns: [DlyCalDt, PERMNO, DlyCap, DlyOpen, DlyHigh, DlyLow, DlyClose, DlyVol]
Index: []


In [1]:
import pandas as pd

# Load CSV
df = pd.read_csv("crsp_top500_daily_1993_2003.csv")

# Make sure date is datetime
df["DlyCalDt"] = pd.to_datetime(df["DlyCalDt"])

# Sort correctly before shift
df = df.sort_values(["PERMNO", "DlyCalDt"]).copy()

# Create 5-day-ahead close price within each stock
df["DlyClose_t_plus_5"] = df.groupby("PERMNO")["DlyClose"].shift(-5)

# Create label:
# 1 if price after 5 trading days is higher than today
# 0 otherwise
df["label"] = (df["DlyClose_t_plus_5"] > df["DlyClose"]).astype("float")

# Drop rows where 5-day-ahead price does not exist
df = df.dropna(subset=["DlyClose_t_plus_5"]).copy()

# Convert label to int after dropping NaNs
df["label"] = df["label"].astype(int)

# Extract year
df["year"] = df["DlyCalDt"].dt.year

# Keep only years 1993 to 2003
df = df[(df["year"] >= 1993) & (df["year"] <= 2003)].copy()

# Calculate yearly percentages
yearly_stats = (
    df.groupby("year")["label"]
    .agg(
        up_pct="mean",
        n_obs="count"
    )
    .reset_index()
)

yearly_stats["down_pct"] = 1 - yearly_stats["up_pct"]

# Convert to percentages
yearly_stats["up_pct"] = yearly_stats["up_pct"] * 100
yearly_stats["down_pct"] = yearly_stats["down_pct"] * 100

# Reorder columns
yearly_stats = yearly_stats[["year", "n_obs", "up_pct", "down_pct"]]

# Print result
print(yearly_stats)


    year   n_obs     up_pct   down_pct
0   1993  126317  50.089062  49.910938
1   1994  125737  46.151888  53.848112
2   1995  125679  54.791970  45.208030
3   1996  126618  51.675117  48.324883
4   1997  126264  54.070836  45.929164
5   1998  125588  52.767780  47.232220
6   1999  125590  49.007883  50.992117
7   2000  125083  50.083545  49.916455
8   2001  123491  50.578585  49.421415
9   2002  125587  47.474659  52.525341
10  2003  123077  55.912153  44.087847


In [2]:
import pandas as pd

df = pd.read_csv("crsp_top500_daily_1993_2003.csv")
df["DlyCalDt"] = pd.to_datetime(df["DlyCalDt"])

print("First date:", df["DlyCalDt"].min())
print("Last date: ", df["DlyCalDt"].max())

First date: 1993-01-04 00:00:00
Last date:  2003-12-31 00:00:00
